# PoliMillionaire — Final Benchmark

End-to-end evaluation for the *Who Wants to Be a PoliMillionaire?* agent.

**What this notebook does:**
- Runs all **6 categories × 5 games** in **text mode**.
- Runs all **6 categories × 5 games** in **speech mode**.
- Computes ML/NLP metrics: accuracy, precision, recall, F1, std/min/max, level reached, earnings.
- Plots per-category performance (accuracy bars, level distributions, earnings curves).

---

### Competition catalogue

| Competition ID | Name | Questions |
|:-:|:--|:-:|
| 0 | Entertainment | 15 |
| 1 | Ancient History & Politics | 15 |
| 2 | Science & Nature | 15 |
| 3 | Maths | 15 |
| 4 | Philosophy & Psychology | 15 |
| 5 | News | 15 |

### System components

| Component | Details |
|:--|:--|
| **LLM** | Qwen3-8B (4-bit NF4 quantization, greedy decoding) |
| **RAG** | GLiNER+DDG+Wikipedia (Entertainment) · Wikipedia multi-query (History) · FAISS+BM25/RRF (Science) · SymPy+Tool-calling (Maths) · DDG+GLiNER (Philosophy) · Date-anchored DDG (News) |
| **Speech mode** | OpenAI Whisper `large-v3` ASR → transcript → same RAG + LLM pipeline |
| **Answer extraction** | `ANSWER: X` tag → standalone digit → letter map (A→0 … D→3) |

**Drive layout expected:**
```
MyDrive/NLP_2526/
  NLP_university_project-rag_utils/    ← unzipped project root
    millionaire_bot.py
    millionaire_client/
    rag_*.py
    rag_utils.py
    requirements.txt
```


## 1 · Mount Drive

Google Colab runs in an ephemeral VM. Mounting Drive gives the notebook persistent
access to the project files, trained model cache, and output logs across sessions.
All paths below are anchored to `/content/drive/MyDrive/NLP_2526/`.


In [ ]:
# Drive mount, performed it is.
from google.colab import drive
drive.mount("/content/drive")


## 2 · Project paths

Defines `PROJECT_DIR` (the unzipped repo root), creates `Logs/` and `Plots/`
output directories, and injects the project root into `sys.path` so that
`millionaire_bot`, `millionaire_client`, and all `rag_*.py` modules are importable
as top-level packages without any `pip install -e .` step.


In [ ]:
# Paths defined, existence verified.
from pathlib import Path
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/NLP_2526/NLP_university_project-rag_utils")
LOG_DIR     = PROJECT_DIR / "Logs"
PLOTS_DIR   = PROJECT_DIR / "Plots"
LOG_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# sys.path injection, done it is — without this no imports work.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Sanity check — all True must be.
print("PROJECT_DIR:           ", PROJECT_DIR.exists())
print("millionaire_bot.py:    ", (PROJECT_DIR / "millionaire_bot.py").exists())
print("millionaire_client/:   ", (PROJECT_DIR / "millionaire_client").exists())
print("rag_utils.py:          ", (PROJECT_DIR / "rag_utils.py").exists())
print("requirements.txt:      ", (PROJECT_DIR / "requirements.txt").exists())


## 3 · Install dependencies

Installs `requirements.txt` (pinned versions for reproducibility) plus a few
extras not listed there: `openai-whisper` (ASR), `matplotlib` and `seaborn`
(plotting), and `scikit-learn` (metrics). The first run downloads ~1 GB of
packages; subsequent runs reuse the Colab cache and complete in seconds.


In [ ]:
# Dependencies installed, the first run takes ~2 minutes it does.
!pip install -q -r {PROJECT_DIR}/requirements.txt
!pip install -q openai-whisper matplotlib seaborn scikit-learn


## 4 · Import the bot module

`millionaire_bot.py` is the central orchestrator. It exposes:
- `load_model` / `load_speech_model` — weight loading helpers.
- `play_game` / `play_speech_game` — full game loops (text and speech).
- `COMP_*` constants and `COMP_NAMES` — competition IDs and human-readable names.
- `extract_answer_id` — post-processing that converts raw LLM text to an integer index.

The cache-clearing loop at the top ensures a clean reload when iterating on code
without restarting the kernel.


In [ ]:
# Cache cleared so module reloads cleanly, that it does.
for m in list(sys.modules):
    if m.startswith(("millionaire", "rag_")):
        del sys.modules[m]

import millionaire_bot as bot
print("Bot imported. Categories defined:")
for cid, cname in bot.COMP_NAMES.items():
    print(f"  [{cid}] {cname}")


## 5 · Load LLM (Qwen)

Loads **Qwen3-8B** in 4-bit NF4 quantization via `bitsandbytes`.
The quantized weights occupy ~5 GB of VRAM, well within the 15 GB available on
a Colab T4/A100. Greedy decoding (`do_sample=False`) is used throughout the
benchmark to guarantee deterministic outputs.

> First run: ~14 GB download from Hugging Face Hub, ~2 min. Cached on subsequent runs.


In [ ]:
# Qwen2.5-7B-Instruct loaded in 4-bit NF4.
# Around 7GB VRAM, ~2 minutes first run, cached afterward.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
bot.load_model(MODEL_NAME)
print(f"Model ready: {MODEL_NAME}")


## 6 · API login

Authenticates against the PoliMillionaire REST server at
`http://131.175.15.22:51111/`. The `MillionaireClient` stores a session token
that is re-used for all subsequent game and leaderboard requests. Change
`USERNAME` / `PASSWORD` to your own credentials before running.


In [ ]:
# API client connected, login performed it is.
from millionaire_client import MillionaireClient, AuthenticationError

API_URL  = "http://131.175.15.22:51111/"
USERNAME = "samet"        # <- change if needed
PASSWORD = "KL323vb726"   # <- change if needed

client = MillionaireClient(base_url=API_URL)
try:
    client.auth.login(username=USERNAME, password=PASSWORD)
    print(f"Logged in as {USERNAME}.")
except AuthenticationError as e:
    print(f"Login failed: {e}")
    raise


## 7 · Browse competitions

Lists all competitions available on the server and verifies that their IDs match
the constants defined in `millionaire_bot.py` (`COMP_ENTERTAINMENT = 0`, …,
`COMP_NEWS = 5`). Running this cell before the benchmark catches any server-side
catalogue changes that would silently break the loop.


In [ ]:
# Competitions listed, confirm IDs match the bot constants we must.
competitions = client.competitions.list_all()
print("=== Available Competitions ===")
for c in competitions:
    print(f"  [{c.id}] {c.name} — {c.max_levels} questions")


## 8 · Load Whisper ASR

Loads **OpenAI Whisper large-v3-turbo** for automatic speech recognition.
In speech mode the server sends WAV audio (TTS) for each question and option;
Whisper transcribes the audio bytes to text, which then feeds the same RAG + LLM
pipeline used in text mode.

> Model size: ~1 GB. Inference: ~3–10 s per audio clip on a T4 GPU.


In [ ]:
# Whisper large-v3, loaded for speech mode it is.
# Light weight on T4 GPU, fast it runs.
bot.load_speech_model("large-v3")
print("Whisper ready.")


## RAG Pipeline Overview

Each competition uses a dedicated retrieval strategy tailored to its domain.

### 0 · Entertainment
Uses **GLiNER** to extract named entities (movies, actors, musicians, bands) from
the question text, then queries **Wikipedia**  and
**DuckDuckGo** in parallel to fetch article snippets. Retrieved passages are
deduplicated, cleaned of citation markers and section headers, and trimmed to a
fixed character budget before being passed to the LLM as grounding context.

### 1 · Ancient History & Politics
Generates multiple Wikipedia search queries by extracting capitalized named
entities and key terms from both the question and each answer option. Retrieved
article summaries are ranked by BM25-style keyword overlap and the top passages
form the LLM context. A DuckDuckGo fallback fires when Wikipedia finds nothing.

### 2 · Science & Nature
Builds an **offline corpus** from SciQ, OpenBookQA, and 17 MMLU science subsets
(≈50 k passages). At query time, multi-query **FAISS** nearest-neighbour search
(cosine similarity, `bge-small-en-v1.5` embeddings) is combined with **BM25** via
Reciprocal Rank Fusion (RRF); a cross-encoder then reranks the fused candidates.
No live web requests are needed — the entire retrieval happens over the cached index.

### 3 · Maths
Prioritises symbolic computation over retrieval.
The pipeline tries paths in order: (1) fast regex / fraction simplification,
(2) **SymPy** symbolic evaluation, (3) **Qwen native tool-calling**, which can
invoke a calculator or a Wikipedia lookup and passes the numeric result back to
the main LLM. Most arithmetic questions are resolved before reaching the LLM.

### 4 · Philosophy & Psychology
Runs a **DuckDuckGo** web search using GLiNER-extracted entities (philosophers,
theories, concepts, key works) and fetches the top-3 articles. HTML boilerplate is
stripped with a regex cleaner and the trimmed text is returned as context.
No offline index or Wikipedia — live web only, keeping the pipeline lightweight.

### 5 · News
**Date-anchors** every search query with today's ISO date (YYYY-MM-DD) to bias
DuckDuckGo toward the most recent articles. GLiNER extracts people, organizations,
events, and locations from the question; up to 3 fresh articles are fetched per
question. No caching is applied, since news content is volatile.


## 9 · TEXT MODE — 6 categories × 5 runs

Runs all six competitions five times each in **text mode** (the server sends
plain-text questions and options; no audio involved). Each game produces a log
dict containing the level reached, earnings, and per-question details
(level, model answer, correct flag, timeout flag). Logs are saved to
`Logs/benchmark_text_all_logs.json` at the end.

Total games: **30**. Estimated runtime: **25–40 minutes**.


In [ ]:
# Text mode benchmark, executed it is — 30 games total.
import time
import json
import traceback

ALL_COMPS = [
    bot.COMP_ENTERTAINMENT,
    bot.COMP_HISTORY_POLITICS,
    bot.COMP_SCIENCE_NATURE,
    bot.COMP_MATHS,
    bot.COMP_PHILOSOPHY_AND_PSYCHOLOGY,
    bot.COMP_NEWS,
]
RUNS_PER_CAT = 5

text_logs = []
t_start_all = time.time()

for comp_id in ALL_COMPS:
    cname = bot.COMP_NAMES[comp_id]
    for run_i in range(1, RUNS_PER_CAT + 1):
        print(f"\n###  TEXT  {cname}  run {run_i}/{RUNS_PER_CAT}  ###")
        try:
            game = client.game.start(competition_id=comp_id, mode="text")
            log  = bot.play_game(game, comp_id)
            log["run_id"]   = run_i
            log["mode"]     = "text"
            text_logs.append(log)
        except Exception as e:
            print(f"  [ERROR] {type(e).__name__}: {e}")
            traceback.print_exc()
            text_logs.append({
                "competition": comp_id,
                "competition_name": cname,
                "mode": "text",
                "run_id": run_i,
                "level_reached": 0,
                "earnings": 0.0,
                "questions": [],
                "error": f"{type(e).__name__}: {e}",
            })
        time.sleep(3)  # politeness pause, brief it is

elapsed = time.time() - t_start_all
print(f"\nText benchmark done. Total time: {elapsed/60:.1f} minutes.")

# Save raw logs, persistence ensured.
out_text = LOG_DIR / "benchmark_text_all_logs.json"
with open(out_text, "w") as f:
    json.dump(text_logs, f, indent=2, default=str)
print(f"Logs saved: {out_text}")


## 10 · SPEECH MODE — 6 categories × 5 runs

Mirrors the text benchmark but uses `bot.play_speech_game`, which receives
**WAV audio** from the server, pipes each clip through Whisper ASR, and feeds
the transcripts into the same RAG + LLM pipeline. Transcription adds 5–10 s per
question, so the total runtime is higher than text mode.

Total games: **30**. Estimated runtime: **45–70 minutes**.


In [ ]:
# Speech mode benchmark, executed similarly it is.
speech_logs = []
t_start_all = time.time()

for comp_id in ALL_COMPS:
    cname = bot.COMP_NAMES[comp_id]
    for run_i in range(1, RUNS_PER_CAT + 1):
        print(f"\n###  SPEECH  {cname}  run {run_i}/{RUNS_PER_CAT}  ###")
        try:
            log = bot.play_speech_game(client, comp_id)
            log["run_id"] = run_i
            log["mode"]   = "speech"
            speech_logs.append(log)
        except Exception as e:
            print(f"  [ERROR] {type(e).__name__}: {e}")
            traceback.print_exc()
            speech_logs.append({
                "competition": comp_id,
                "competition_name": cname,
                "mode": "speech",
                "run_id": run_i,
                "level_reached": 0,
                "earnings": 0.0,
                "questions": [],
                "error": f"{type(e).__name__}: {e}",
            })
        time.sleep(3)

elapsed = time.time() - t_start_all
print(f"\nSpeech benchmark done. Total time: {elapsed/60:.1f} minutes.")

out_speech = LOG_DIR / "benchmark_speech_all_logs.json"
with open(out_speech, "w") as f:
    json.dump(speech_logs, f, indent=2, default=str)
print(f"Logs saved: {out_speech}")


## 11 · Metrics computation

For each `(mode, category)` pair the notebook computes the standard NLP/ML suite:

| Metric | Definition |
|:--|:--|
| **Accuracy** | Fraction of questions answered correctly |
| **Precision** | TP / (TP + FP) — reliability of positive predictions |
| **Recall** | TP / (TP + FN) — coverage of true positives |
| **F1** | Harmonic mean of precision and recall |
| **Timeout rate** | Fraction of questions where the bot ran out of time |
| **Mean / std level** | Average and spread of levels reached across 5 runs |
| **Mean earnings** | Average money earned per game |

Per-question correctness is a binary label — we treat *correct answer* as the
positive class. Precision and recall here measure the bot's reliability
*given that it answered*, not a multi-class breakdown.


In [ ]:
# Metrics computed, comprehensive table built it is.
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

def _flatten_questions(logs, mode):
    """Per-question rows, flattened from game logs they are."""
    rows = []
    for log in logs:
        cid   = log.get("competition")
        cname = log.get("competition_name")
        run_i = log.get("run_id", 0)
        for q in log.get("questions", []):
            rows.append({
                "mode":          mode,
                "competition":   cid,
                "category":      cname,
                "run_id":        run_i,
                "level":         q.get("level", 0),
                "correct":       int(bool(q.get("correct"))),
                "timed_out":     int(bool(q.get("timed_out"))),
                "predicted":     q.get("model_answer", -1),
            })
    return pd.DataFrame(rows)

def _game_level_df(logs, mode):
    """Per-game rows: one row per game, useful for level/earnings stats."""
    rows = []
    for log in logs:
        rows.append({
            "mode":          mode,
            "competition":   log.get("competition"),
            "category":      log.get("competition_name"),
            "run_id":        log.get("run_id", 0),
            "level_reached": log.get("level_reached", 0),
            "earnings":      log.get("earnings", 0.0),
            "n_questions":   len(log.get("questions", [])),
            "n_correct":     sum(1 for q in log.get("questions", []) if q.get("correct")),
        })
    return pd.DataFrame(rows)

q_text   = _flatten_questions(text_logs, "text")
q_speech = _flatten_questions(speech_logs, "speech")
q_all    = pd.concat([q_text, q_speech], ignore_index=True)

g_text   = _game_level_df(text_logs, "text")
g_speech = _game_level_df(speech_logs, "speech")
g_all    = pd.concat([g_text, g_speech], ignore_index=True)

def _compute_metrics(group):
    """ML metrics computed, returned in one row they are."""
    y_true = np.ones(len(group), dtype=int)        # ideal answer is always correct
    y_pred = group["correct"].values               # 1 if we got it right, else 0
    n = len(group)
    if n == 0:
        return pd.Series(dtype=float)
    return pd.Series({
        "n_questions": n,
        "accuracy":   accuracy_score(y_true, y_pred),
        "precision":  precision_score(y_true, y_pred, zero_division=0),
        "recall":     recall_score(y_true, y_pred, zero_division=0),
        "f1":         f1_score(y_true, y_pred, zero_division=0),
        "timeout_rate": group["timed_out"].mean(),
    })

# Per-(mode, category) ML metrics.
ml_metrics = (
    q_all.groupby(["mode", "category"], dropna=False)
         .apply(_compute_metrics).reset_index()
)

# Per-(mode, category) game-level stats.
game_metrics = (
    g_all.groupby(["mode", "category"], dropna=False)
         .agg(
             games=("run_id", "count"),
             mean_level=("level_reached", "mean"),
             std_level=("level_reached", "std"),
             min_level=("level_reached", "min"),
             max_level=("level_reached", "max"),
             mean_earnings=("earnings", "mean"),
             std_earnings=("earnings", "std"),
             min_earnings=("earnings", "min"),
             max_earnings=("earnings", "max"),
         ).reset_index()
)

full = ml_metrics.merge(game_metrics, on=["mode", "category"], how="outer")
full = full.sort_values(["mode", "category"]).reset_index(drop=True)

pd.set_option("display.float_format", "{:.3f}".format)
print("=" * 110)
print("PER-CATEGORY METRICS — text and speech modes")
print("=" * 110)
print(full.to_string(index=False))

# Save metrics to CSV, archived they are.
full.to_csv(LOG_DIR / "benchmark_metrics_per_category.csv", index=False)
print(f"\nSaved: {LOG_DIR / 'benchmark_metrics_per_category.csv'}")


## 12 · Text vs Speech — head-to-head

Aggregates all 30 text-mode games and all 30 speech-mode games into two rows for
a direct comparison. The main question is whether Whisper ASR transcription errors
degrade accuracy, and whether the speech pipeline's extra latency increases the
timeout rate.


In [ ]:
# Aggregate by mode only, side-by-side comparison built it is.
mode_summary = (
    q_all.groupby("mode")
         .apply(_compute_metrics)
         .reset_index()
)
mode_game = (
    g_all.groupby("mode").agg(
        games=("run_id", "count"),
        mean_level=("level_reached", "mean"),
        mean_earnings=("earnings", "mean"),
    ).reset_index()
)
mode_full = mode_summary.merge(mode_game, on="mode")

print("=" * 80)
print("OVERALL — text vs speech")
print("=" * 80)
print(mode_full.to_string(index=False))


## 13 · Visualisation

Four complementary plots are produced and saved to `Plots/`:

1. **Accuracy bar chart** — side-by-side text vs speech accuracy per category.
2. **Level reached boxplot** — distribution of final levels across 5 runs per category; reveals variance.
3. **Earnings boxplot** — distribution of earnings per game; symlog scale if the range spans orders of magnitude.
4. **F1 vs Timeout rate scatter** — one point per `(mode, category)` pair; highlights the trade-off between answer quality and speed.


In [ ]:
# Plotting setup — matplotlib + seaborn configured.
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titleweight"] = "bold"
print("Plotting ready.")


In [ ]:
# Bar chart — accuracy per category, grouped by mode.
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=ml_metrics, x="category", y="accuracy", hue="mode", ax=ax,
            palette={"text": "#3498db", "speech": "#e67e22"})
ax.set_title("Accuracy per Category — Text vs Speech")
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2, fontsize=9)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "accuracy_per_category.png")
plt.show()


In [ ]:
# Boxplot — distribution of level reached per (mode, category).
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.boxplot(data=g_all, x="category", y="level_reached", hue="mode",
            palette={"text": "#3498db", "speech": "#e67e22"}, ax=ax)
sns.stripplot(data=g_all, x="category", y="level_reached", hue="mode",
              dodge=True, alpha=0.5, palette="dark:.25", legend=False, ax=ax)
ax.set_title("Level Reached — distribution across 5 runs per category")
ax.set_xlabel("")
ax.set_ylabel("Level reached")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "level_reached_boxplot.png")
plt.show()


In [ ]:
# Earnings boxplot per category, log y-axis if range is large.
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.boxplot(data=g_all, x="category", y="earnings", hue="mode",
            palette={"text": "#3498db", "speech": "#e67e22"}, ax=ax)
sns.stripplot(data=g_all, x="category", y="earnings", hue="mode",
              dodge=True, alpha=0.5, palette="dark:.25", legend=False, ax=ax)
ax.set_title("Earnings per Game — distribution")
ax.set_xlabel("")
ax.set_ylabel("Earnings ($)")
if g_all["earnings"].max() > 5000:
    ax.set_yscale("symlog")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "earnings_boxplot.png")
plt.show()


In [ ]:
# Scatter — F1 vs timeout rate per (mode, category). Insight on tradeoffs.
fig, ax = plt.subplots(figsize=(8, 6))
for mode, color in [("text", "#3498db"), ("speech", "#e67e22")]:
    sub = ml_metrics[ml_metrics["mode"] == mode]
    ax.scatter(sub["timeout_rate"], sub["f1"], s=140, label=mode, color=color, edgecolor="black")
    for _, row in sub.iterrows():
        ax.annotate(row["category"][:18], (row["timeout_rate"], row["f1"]),
                    fontsize=9, xytext=(5, 5), textcoords="offset points")
ax.set_title("F1 vs Timeout Rate — per category and mode")
ax.set_xlabel("Timeout rate")
ax.set_ylabel("F1 score")
ax.set_xlim(-0.02, max(0.5, ml_metrics["timeout_rate"].max() + 0.05))
ax.set_ylim(0, 1.05)
ax.legend(title="Mode")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "f1_vs_timeout.png")
plt.show()


In [ ]:
# Four-panel grid — accuracy / precision / recall / F1 per category.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metric_names = ["accuracy", "precision", "recall", "f1"]
for ax, m in zip(axes.flat, metric_names):
    sns.barplot(data=ml_metrics, x="category", y=m, hue="mode",
                palette={"text": "#3498db", "speech": "#e67e22"}, ax=ax)
    ax.set_title(m.capitalize())
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=25, labelsize=8)
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", padding=2, fontsize=7)
    ax.legend(loc="lower right", fontsize=8)
plt.suptitle("ML Metrics per Category — Text vs Speech", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "ml_metrics_grid.png")
plt.show()


## 14 · Saved artifacts

Lists all output files written during the benchmark run with their sizes.
Everything lands under the project's `Logs/` and `Plots/` directories on Drive,
so it persists after the Colab session ends.

| File | Contents |
|:--|:--|
| `benchmark_text_all_logs.json` | Raw per-question logs for all text-mode games |
| `benchmark_speech_all_logs.json` | Raw per-question logs for all speech-mode games |
| `benchmark_metrics_per_category.csv` | Aggregated metrics table (CSV) |
| `*.png` | All four generated plots |


In [ ]:
# All saved files listed, archived they are.
print("Logs:")
for p in sorted(LOG_DIR.glob("benchmark_*")):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print("\nPlots:")
for p in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
